# 01 · Coleta → camada bronze

**MVP — Onde subsidiar o frete no e-commerce brasileiro: pipeline de dados em nuvem (Databricks)**

Universidade de Brasília (UnB) · Departamento de Engenharia de Produção · Sistemas de Suporte à Decisão
Prof. André Luiz Marques Serrano · **Aluna: Brunna Gavião Moraes — 231025341**


Este notebook cria a estrutura na nuvem e coleta os dados **sem alterá-los**:

1. cria no Unity Catalog os schemas `frete_bronze`, `frete_silver`, `frete_gold` e o Volume `frete_bronze.arquivos`;
2. baixa os 9 arquivos da Olist (Kaggle) e a população por UF do IBGE (API SIDRA);
3. grava tudo na camada bronze, em uma pasta datada (`coleta=AAAA-MM-DD`), com o hash SHA-256 de cada arquivo.

| Fonte | Conteúdo | Licença | Acesso |
|---|---|---|---|
| [Brazilian E-Commerce Public Dataset by Olist](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce) | ~100 mil pedidos (2016–2018), 9 CSVs | CC BY-NC-SA 4.0 | API pública do Kaggle (`kagglehub`) |
| [IBGE — SIDRA, tabela 6579](https://sidra.ibge.gov.br/tabela/6579) | População residente estimada por UF, 2018 | Dados públicos (Lei de Acesso à Informação) | API SIDRA |

In [0]:
%pip install -q kagglehub statsmodels

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
# Configuração comum: localiza o repositório, importa o módulo de apoio e detecta a plataforma.
# No Databricks a variável `spark` já existe; fora dele o mesmo código roda localmente (Parquet + DuckDB).
import os, sys, importlib
from pathlib import Path

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ / "scripts"))
import mvp_frete as mvp
importlib.reload(mvp)

import pandas as pd
pd.set_option("display.max_columns", 50, "display.width", 200)

plat = mvp.Plataforma(spark if "spark" in globals() else None)
print("Ambiente :", plat.preparar())
print("Bronze   :", plat.dir_bronze)
print("Evidências:", plat.dir_evidencias)

Ambiente : Databricks — catálogo workspace, schemas frete_bronze, frete_silver, frete_gold
Bronze   : /Volumes/workspace/frete_bronze/arquivos
Evidências: /Workspace/Users/brunnagmoraes17@gmail.com/MVP3-SSD-Frete-Olist/evidencias


## 1. Pasta da coleta

Cada execução grava em uma pasta própria, datada. Assim a coleta fica rastreável e uma nova coleta não apaga a anterior.

In [0]:
import datetime as dt, shutil, json

data_coleta = dt.date.today().isoformat()
dir_coleta = plat.dir_bronze / f"coleta={data_coleta}"
dir_coleta.mkdir(parents=True, exist_ok=True)
dir_upload = plat.dir_bronze / "upload_manual"   # plano B: arquivos enviados pela interface
print(dir_coleta)

/Volumes/workspace/frete_bronze/arquivos/coleta=2026-11-09


## 2. Olist (Kaggle)

O download usa a API pública do Kaggle, sem login. **Plano B:** se o ambiente bloquear o acesso à internet, envie os 9 CSVs
pela interface (*Catalog → workspace → frete_bronze → arquivos → Upload to this volume*) para a pasta `upload_manual/`
e execute a célula de novo.

In [0]:
origem = None
try:
    import kagglehub
    origem = Path(kagglehub.dataset_download(mvp.KAGGLE_DATASET))
    forma = f"kagglehub ({mvp.KAGGLE_DATASET}, pasta {origem.name})"
except Exception as erro:
    print("Download pelo Kaggle indisponível:", type(erro).__name__, erro)
    if all((dir_upload / a).exists() for a in mvp.ARQUIVOS_OLIST):
        origem, forma = dir_upload, "upload manual pela interface do Databricks"
    else:
        raise RuntimeError(f"Envie os 9 CSVs da Olist para {dir_upload} e execute novamente.")

for arq in mvp.ARQUIVOS_OLIST:
    shutil.copy(origem / arq, dir_coleta / arq)
print("Olist copiada para a bronze via", forma)

100%|██████████| 42.6M/42.6M [00:00<00:00, 83.9MB/s]

Extracting files...


Olist copiada para a bronze via kagglehub (olistbr/brazilian-ecommerce, pasta 2)


## 3. IBGE — população residente estimada por UF (2018)

In [0]:
try:
    mvp.baixar_ibge_populacao(dir_coleta / mvp.ARQUIVO_IBGE)
    forma_ibge = f"API SIDRA ({mvp.IBGE_URL})"
except Exception as erro:
    print("API do IBGE indisponível:", erro, "→ usando a cópia versionada em dados/")
    shutil.copy(RAIZ / "dados" / mvp.ARQUIVO_IBGE, dir_coleta / mvp.ARQUIVO_IBGE)
    forma_ibge = "cópia versionada no repositório (dados/ibge_populacao_uf_2018.csv)"
pd.read_csv(dir_coleta / mvp.ARQUIVO_IBGE).head()

,codigo_ibge,uf_nome,ano,populacao
0,11,Rondônia,2018,1757589
1,12,Acre,2018,869265
2,13,Amazonas,2018,4080611
3,14,Roraima,2018,576568
4,15,Pará,2018,8513497


## 4. Metadados de reprodutibilidade

Data da coleta, forma de acesso, volume (linhas, colunas, MB) e **hash SHA-256** de cada arquivo. Quem repetir a
coleta pode comparar os hashes e confirmar que recebeu exatamente os mesmos arquivos.

In [0]:
meta = [mvp.metadados_arquivo(dir_coleta / a, "Kaggle — Olist") for a in mvp.ARQUIVOS_OLIST]
meta.append(mvp.metadados_arquivo(dir_coleta / mvp.ARQUIVO_IBGE, "IBGE — SIDRA 6579"))
metadados = {
    "data_coleta": dt.datetime.now().isoformat(timespec="seconds"),
    "ambiente": plat.ambiente(),
    "pasta_bronze": str(dir_coleta),
    "forma_coleta_olist": forma,
    "forma_coleta_ibge": forma_ibge,
    "licenca_olist": "CC BY-NC-SA 4.0",
    "arquivos": meta,
}
(dir_coleta / "_metadados_coleta.json").write_text(json.dumps(metadados, indent=2, ensure_ascii=False), encoding="utf-8")
(plat.dir_evidencias / "metadados").mkdir(parents=True, exist_ok=True)
(plat.dir_evidencias / "metadados" / "_metadados_coleta.json").write_text(
    json.dumps(metadados, indent=2, ensure_ascii=False), encoding="utf-8")

tab_meta = pd.DataFrame(meta)
print(f"{len(tab_meta)} arquivos · {tab_meta.linhas.sum():,} linhas · {tab_meta.mb.sum():.2f} MB".replace(",", "."))
display(tab_meta[["arquivo", "fonte", "linhas", "colunas", "mb", "sha256"]])

10 arquivos · 1.550.949 linhas · 120.35 MB


arquivo,fonte,linhas,colunas,mb,sha256
olist_orders_dataset.csv,Kaggle — Olist,99441,8,16.84,8df58ef3d2d7e9944010f7beecd9b75367f5588ec6e3c91cec19ae3345ef9ecf
olist_order_items_dataset.csv,Kaggle — Olist,112650,7,14.72,0bc4d068c4fe38cbb01bd90e8746e3c613fe7b4baef75fab7b0e329701c3e279
olist_customers_dataset.csv,Kaggle — Olist,99441,5,8.62,983a422239e1712ded753b3bf9ecf47dc73f144d306029dcfa99e70a226883d2
olist_sellers_dataset.csv,Kaggle — Olist,3095,4,0.17,1f643d2b950373b85735e7794b20986f528d7a000432e7c6f9bcbb44d0846a0e
olist_products_dataset.csv,Kaggle — Olist,32951,9,2.27,3e6569628a17fbc75fd206ee357b59e20364b9afa90f5b6cd5b4d624c58aa9cc
olist_order_reviews_dataset.csv,Kaggle — Olist,99224,7,13.78,012b61c7593e34f51fa614efdf802b9c7056ce6aae5307ddb93236e7cfc797d7
olist_order_payments_dataset.csv,Kaggle — Olist,103886,5,5.51,4f713964f2815dbbaa40b9488268c55aac3627bfce5aa96cf58d1f3616de3cc0
olist_geolocation_dataset.csv,Kaggle — Olist,1000163,5,58.44,b514f6fc991b9566aeba02aa5d67e2c3630f034b60a0e05aa0d082a3b66d88d6
product_category_name_translation.csv,Kaggle — Olist,71,2,0.0,a81f0d1f27b27e7293f761bc79e3ce8f348ee39c4b3ed3e49bde38f478586278
ibge_populacao_uf_2018.csv,IBGE — SIDRA 6579,27,4,0.0,06421a1a8737bb7dc79a47d74ba871d5e8dd3c6c733965b5ed3d6d374f1680e5


In [0]:
# Conferência: o que está gravado na nuvem (listagem da pasta da coleta na bronze)
display(pd.DataFrame([{"arquivo": p.name, "bytes": p.stat().st_size} for p in sorted(dir_coleta.iterdir())]))

arquivo,bytes
_metadados_coleta.json,3112
ibge_populacao_uf_2018.csv,761
olist_customers_dataset.csv,9033957
olist_geolocation_dataset.csv,61273883
olist_order_items_dataset.csv,15438671
olist_order_payments_dataset.csv,5777138
olist_order_reviews_dataset.csv,14451670
olist_orders_dataset.csv,17654914
olist_products_dataset.csv,2379446
olist_sellers_dataset.csv,174703


> **Evidência a capturar:** *Catalog Explorer → workspace → frete_bronze → Volumes → arquivos → coleta=…* mostrando os
> 10 arquivos e o `_metadados_coleta.json` (salvar como `evidencias/01_volume_bronze.png`).